# Lesson 02 Lab — Tensor Core Constraints for Low-Precision GEMM

**Puzzle:** A low-precision dtype is available, so will every matrix multiplication automatically become a fast Tensor Core operation?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A peak-TFLOPS table describes a capability of the chip, not the path selected for every matrix multiplication. In an LLM, the same nominal BF16 operation can arrive with different M, N, and K dimensions, strides, transpositions, and batch sizes. Those details determine whether useful work fills the matrix-multiply tiles or whether edge handling, memory traffic, and launch overhead dominate.


## 0. Predict before running

1. Predict whether BF16 will beat FP32 for both shapes, then predict which shape will lose more efficiency.
2. State what timing can prove and what extra trace would be required before naming a Tensor Core instruction.
3. Choose the shape information that must be preserved for another reader to reproduce the result.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A GEMM consumes `A[M,K]` and `B[K,N]`. Dtype, strides, transposition, leading dimensions, and the three logical sizes travel together into dispatch; the word *BF16* by itself is not a kernel description.

- A dtype is only one dispatch condition; layout, dimensions, alignment, and backend policy also select the kernel.
- Arithmetic intensity separates compute-bound GEMMs from shapes dominated by memory traffic or launch overhead.
- Timing establishes performance for a shape; operator or kernel evidence establishes what ran.


## 2. Derive the mechanism

A useful first model is `FLOPs ≈ 2MKN` and `arithmetic intensity = FLOPs / bytes moved`. Large aligned tiles can amortize loads and feed matrix-multiply hardware; awkward dimensions create edge tiles, padding, or a different implementation. Tensor Core eligibility is therefore a conjunction of hardware, dtype, shape, layout, and library support.

For `C[M,N] = A[M,K] @ B[K,N]`, the leading operation count is `2MKN`. That number is only the numerator of the performance story. A first roofline estimate divides it by bytes moved; a dispatch estimate also asks whether M, N, K, layout, alignment, and dtype fit a library kernel's tiling rules. When `N=2055`, the mathematical work increases by only about 0.34% relative to `N=2048`, yet the physical implementation may need a tail tile or a different kernel. A large timing discontinuity is therefore evidence about shape sensitivity, not proof of one particular instruction.

This distinction matters in attention and MLP layers because their matrices are not interchangeable. Prefill creates large M dimensions, while Decode often presents GEMV-like or very small-M work. A kernel that is excellent for one phase can leave Tensor Cores under-filled in another. The useful unit of reasoning is consequently a shape family plus an operator trace, not the model's advertised precision.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "02-tensor-core-constraints"
device = require_cuda()
torch.manual_seed(2026 + 2)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | FP32 GEMM for the exact aligned and awkward shapes |
| Candidate | BF16 GEMM for the same tensors and timing protocol |
| Held constant | GPU, M and K, random distribution, warm-up, repetitions, CUDA-event timing |
| Measurements | median and p90 latency for each dtype/shape pair |
| Evidence | `pytorch-gpu` |

**Experiment:** Time FP32 and BF16 matrix multiplications with aligned and deliberately awkward dimensions on the same GPU.


## 5. Read the experiment code

The lab changes dtype and one alignment condition while keeping the GPU and timing method fixed; the output is shape evidence, not a native-kernel assertion.

The notebook allocates each shape once, warms the operation four times, and records twelve CUDA-event samples. Synchronization happens inside the timing helper so host launch latency is not mistaken for completed GPU work. The aligned and awkward cases differ only in N; this keeps the comparison narrow enough to attribute a timing change to shape and dispatch behavior.

The code deliberately does not parse native kernel names. PyTorch-level timing tells us what the application observed, while Nsight Systems or Nsight Compute would be the next evidence layer for `mma`/Tensor Core utilization, tile occupancy, memory throughput, and tail effects.

Only after these variables match the protocol should the cell be executed.


In [2]:
shapes = {"aligned": (2048, 2048, 2048), "awkward": (2048, 2055, 2048)}
timings = {}
for name, (m, k, n) in shapes.items():
    timings[name] = {}
    for dtype in (torch.float32, torch.bfloat16):
        a = torch.randn(m, k, device=device, dtype=dtype)
        b = torch.randn(k, n, device=device, dtype=dtype)
        timings[name][str(dtype).split(".")[-1]] = cuda_benchmark(lambda: a @ b, warmup=4, repeats=12)
result = base_result(2, "pytorch-gpu")
result.update({"shapes_mkn": shapes, "timings": timings,
               "conclusion": "Observed shape- and dtype-dependent GEMM timing; native Tensor Core identity requires a lower-level profiler."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Aligned BF16 median | 0.087632 ms |
| Aligned FP32 median | 0.262096 ms |
| Awkward BF16 median | 0.176048 ms |
| Awkward FP32 median | 0.319312 ms |
| Recorded samples per case | 12 |


## 7. Interpret rather than merely print

On the checked-in RTX 5090 run, aligned BF16 took 0.087632 ms versus 0.262096 ms for FP32, a 2.99x ratio. Changing only N from 2048 to 2055 raised BF16 latency to 0.176048 ms—about 2.01x the aligned BF16 time—even though the arithmetic count barely changed. FP32 also slowed, but by a smaller 1.22x ratio.

The correct conclusion is not that `2055` is universally bad or that one named Tensor Core kernel was missed. It is that dtype speedups are conditional on shape, and that an awkward boundary can erase a large fraction of the expected benefit. Native dispatch identity remains an explicit follow-up measurement.

**Inspection rule:** Compare medians by dtype and shape. The lab does not infer Tensor Core use from speed alone; it records a PyTorch GPU timing baseline for later profiler work.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Observed shape- and dtype-dependent GEMM timing; native Tensor Core identity requires a lower-level profiler.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:07+00:00",
  "lesson": 2,
  "schema_version": 1,
  "shapes_mkn": {
    "aligned": [
      2048,
      2048,
      2048
    ],
    "awkward": [
      2048,
      2055,
      2048
    ]
  },
  "timings": {
    "aligned": {
      "bfloat16": {
        "median_ms": 0.087632,
        "p90_ms": 0.088864,
        "repeats": 12,
        "samples_ms": [
          0.0936,
          0.08896,
          0.08768,
          0.087584,
          0.086816,
          0.087904,
          0.086752,
          0.086528,
          0.088864,
          0.087104,
          0.087872,
          0.087456
        ],
 

## 9. Make the bounded decision

> Low precision creates an opportunity, not a guarantee. Preserve exact shapes and profiler evidence when deciding whether a Tensor Core path was reached.

**Acceptance/rollback:** Keep the exact `M,N,K`, strides, dtype, warm-up, and repeated timing. Use an operator trace to show dispatch and Nsight Compute/System metrics before naming a native Tensor Core kernel.

**Failure analysis:** A misleading benchmark would compare different shapes, include first-call initialization, report one sample, or infer Tensor Core use from a fast BF16 result. Padding is also not automatically a fix: it may improve tile utilization while adding FLOPs and temporary storage. Accept padding only after measuring the complete padded operator and its downstream layout costs.


## 10. Extend the evidence

Profile both shapes with Nsight Compute and record the selected kernel, achieved occupancy, tensor-pipe utilization, DRAM throughput, and wasted edge work. Then repeat with Decode-like M values such as 1, 8, and 32. The exercise is successful when you can explain a reversal using both the trace and the timing distribution rather than the dtype label alone.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
